<a href="https://colab.research.google.com/github/Priya-Kumari-Chourasia/Pytorch_tutorials/blob/main/pytorch_training_pipeline_using_dataset_and_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/breast-cancer-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'breast-cancer-dataset' dataset.
Path to dataset files: /kaggle/input/breast-cancer-dataset


In [5]:
df = pd.read_csv(f"{path}/breast-cancer.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
df.drop(columns=['id'],inplace=True)

In [7]:
x_train,x_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [8]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [9]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [10]:
x_train_tensor = torch.from_numpy(x_train)
x_test_tensor = torch.from_numpy(x_test)

y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [11]:
class cust(Dataset):
  def __init__(self,features,labels):
    self.features=features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self,idx):
    return self.features[idx],self.labels[idx]

In [12]:
train_dataset=cust(x_train_tensor,y_train_tensor)
test_dataset = cust(x_test_tensor,y_test_tensor)

In [13]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

#### Defining the model

In [14]:
class simplemod(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear = nn.Linear(num_features,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):
    out = self.linear(features)
    out = self.sigmoid(out)

    return out

In [15]:
learning_rate = 0.1
epochs = 25

In [16]:
model = simplemod(x_train_tensor.shape[1]).double()

optimizer = torch.optim.SGD(model.parameters(),lr = learning_rate)

loss_function = nn.BCELoss()

In [17]:
for epoch in range(epochs):
  for batch_features,batch_labels in train_loader:
    y_pred = model(batch_features)

    loss = loss_function(y_pred, batch_labels.view(-1,1).double())

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(f'Epoch:{epoch+1},Loss : {loss.item()}')



Epoch:1,Loss : 0.6521692210133638
Epoch:1,Loss : 0.5166566740198546
Epoch:1,Loss : 0.4533196337530554
Epoch:1,Loss : 0.41586267098434915
Epoch:1,Loss : 0.3483663507469886
Epoch:1,Loss : 0.34653481803658037
Epoch:1,Loss : 0.34659605672476784
Epoch:1,Loss : 0.2209695185128365
Epoch:1,Loss : 0.2800691392654865
Epoch:1,Loss : 0.30044743782858685
Epoch:1,Loss : 0.1406602593832556
Epoch:1,Loss : 0.20847233587102756
Epoch:1,Loss : 0.1814445750019716
Epoch:1,Loss : 0.20115195177655512
Epoch:1,Loss : 0.19152077235887208
Epoch:2,Loss : 0.21247660574262833
Epoch:2,Loss : 0.2147177082286459
Epoch:2,Loss : 0.21663151159886623
Epoch:2,Loss : 0.16348301095175302
Epoch:2,Loss : 0.2346688165635213
Epoch:2,Loss : 0.12623720324837712
Epoch:2,Loss : 0.13144531184398067
Epoch:2,Loss : 0.16009869405223043
Epoch:2,Loss : 0.15843158824581385
Epoch:2,Loss : 0.17718104752168803
Epoch:2,Loss : 0.1706552764171621
Epoch:2,Loss : 0.1216572391488357
Epoch:2,Loss : 0.16669982493831256
Epoch:2,Loss : 0.267395200479598

In [18]:
model.eval()
accuracy_list = []

with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    y_pred = model(batch_features)
    y_pred = (y_pred >0.8).float()

    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

overall_accuracy = sum(accuracy_list)/len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')

Accuracy: 0.9470
